<a href="https://colab.research.google.com/github/Hemant10HM/ANNDL-LAB_24mcs004/blob/main/ANN_LAB_7_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Implementation of BERT on SSt2 data set


In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.optim import Adam
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from datasets import load_dataset
from tqdm import tqdm

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load SST-2 dataset
dataset = load_dataset("glue", "sst2")
print("Dataset loaded successfully.")

# Load RoBERTa tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
model.to(device)

# Tokenization function
def tokenize_function(example):
    return tokenizer(example["sentence"], padding="max_length", truncation=True, max_length=128)

# Tokenize dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenized.")

# Set format for PyTorch
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
print("Dataset format set to PyTorch.")

# Create DataLoaders
train_loader = DataLoader(tokenized_datasets["train"], batch_size=32, shuffle=True)
val_loader = DataLoader(tokenized_datasets["validation"], batch_size=32)
print("DataLoaders created.")

# Optimizer (using Adam from torch.optim)
optimizer = Adam(model.parameters(), lr=2e-5)

# Training loop
num_epochs = 2
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} Training")

    for batch in progress_bar:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

        progress_bar.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} Training Loss: {total_loss / len(train_loader):.4f}")

    # Evaluation loop
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=-1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    print(f"Validation Accuracy after epoch {epoch+1}: {accuracy:.4f}")


c:\Users\heman\Desktop\Hemant Study\LNMIIT\Python Projects Environments\plant\plant\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Dataset loaded successfully.


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Dataset tokenized.
Dataset format set to PyTorch.
DataLoaders created.


Epoch 1 Training: 100%|██████████| 2105/2105 [2:09:16<00:00,  3.68s/it, loss=0.0253]  


...

Validation Accuracy after epoch 1: 0.9300


Epoch 2 Training: 100%|██████████| 2105/2105 [2:09:14<00:00,  3.68s/it, loss=0.111]   


Epoch 2 Training Loss: 0.1407


Evaluating: 100%|██████████| 28/28 [00:53<00:00,  1.92s/it]

Validation Accuracy after epoch 2: 0.9404


#Here I did ot use testing but validation for eval so the final accuracy is 94%
